# 模型

## 课程目标
* 了解各种 Claude 模型
* 比较 Claude 模型的速度和能力


让我们开始导入 `anthropic` SDK 并加载我们的 API 密钥：

In [22]:
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()

## Claude 模型

Claude Python SDK 支持多种模型，每种模型具有不同的能力和性能特点。此图表比较了 Claude 3 和 3.5 模型之间的成本与速度，展示了成本与智能之间的权衡范围：



选择模型时，需要考虑以下几个重要因素：

* 模型的延迟（它有多快？）
* 模型的能力（它有多智能？）
* 模型的成本（它有多贵？）



请参阅[此表格](https://docs.anthropic.com/en/docs/about-claude/models#model-comparison-table)比较 Claude 系列中每个模型的关键功能和能力。

## 比较模型速度

下面是一个简单的函数，它对所有 4 个模型运行相同的提示词，并打印出模型响应和每个请求所花费的时间。

In [3]:
import time
def compare_model_speeds():
    models = ["claude-3-5-sonnet-20240620","claude-3-opus-20240229", "claude-3-sonnet-20240229", "claude-3-haiku-20240307"]
    task = "Explain the concept of photosynthesis in a concise paragraph."

    for model in models:
        start_time = time.time()

        response = client.messages.create(
            model=model,
            max_tokens=500,
            messages=[{"role": "user", "content": task}]
        )

        end_time = time.time()
        execution_time = end_time - start_time
        tokens = response.usage.output_tokens
        time_per_token = execution_time/tokens

        print(f"Model: {model}")
        print(f"Response: {response.content[0].text}")
        print(f"Generated Tokens: {tokens}")
        print(f"Execution Time: {execution_time:.2f} seconds")
        print(f"Time Per Token: {time_per_token:.2f} seconds\n")

In [4]:
compare_model_speeds()

Model: claude-3-5-sonnet-20240620
Response: Photosynthesis is the process by which plants, algae, and some bacteria convert light energy into chemical energy. These organisms use sunlight, water, and carbon dioxide to produce glucose (a type of sugar) and oxygen. The process occurs primarily in the chloroplasts of plant cells, where chlorophyll, a green pigment, absorbs light energy. This energy is then used to drive a series of chemical reactions that ultimately result in the production of glucose, which serves as food for the plant and can be stored for later use. Oxygen is released as a byproduct of this process, making photosynthesis crucial for maintaining Earth's atmosphere and supporting life on the planet.
Generated Tokens: 146
Execution Time: 2.56 seconds
Time Per Token: 0.02 seconds

Model: claude-3-opus-20240229
Response: Photosynthesis is the process by which green plants and some other organisms use sunlight to synthesize nutrients from carbon dioxide and water. In plants,

运行上述代码时获得的确切响应会有所不同，但以下是运行上述代码时获得的特定输出的表格总结：


| 模型 | 生成的 Token 数 | 执行时间（秒）| 每 Token 时间（秒）|
|-------|------------------|--------------------------|--------------------------|
| claude-3-5-sonnet-20240620 | 146 | 2.56 | 0.02 |
| claude-3-opus-20240229 | 146 | 7.32 | 0.05 |
| claude-3-sonnet-20240229 | 108 | 2.64 | 0.02 |
| claude-3-haiku-20240307 | 126 | 1.09 | 0.01 |

同样重要的是要注意，对于像"用简洁的段落解释光合作用的概念"这样简单的提示词，所有模型都表现良好。在这种情况下，选择最快和最便宜的选项可能是最合理的。



上面的例子简单说明了模型速度差异，但这不是一个非常严格的演示。这是我们通过向所有 3 个模型提供相同的输入提示词 50 次并平均每个模型的响应时间生成的图表。为了确保"公平"比较，我们提示模型生成非常长的输出，然后使用 `max_tokens` 将所有模型响应截断到完全相同的 Token 数量（我们将在下一课中介绍这一点）。



## 比较模型能力

显然 Haiku 是最快的模型，那么我们为什么还要使用其他模型呢？这完全取决于模型速度、成本和整体能力之间的权衡。Haiku 是最快的，但在某些情况下，它的输出质量可能不如 Opus。话虽如此，需要注意的是，在许多情况下，Haiku 的表现可以与我们一些更强大的模型相媲美。唯一能真正知道哪个模型是您的特定用例的"最佳"选择的方法是尝试它们并评估它们的性能。

一般来说，我们建议将我们最强大的模型 Claude 3.5 Sonnet 用于以下用例：
* **编码：** Claude 3.5 Sonnet 可以自主编写、编辑和运行代码，简化代码翻译，实现更快、更准确的更新和迁移。
* **客户支持：** Claude 3.5 Sonnet 理解用户上下文并协调多步骤工作流程，实现 24/7 全天候支持、更快的响应和更高的客户满意度。
* **数据科学与分析：** Claude 3.5 Sonnet 导航非结构化数据，生成洞察，并产生可视化和预测以增强数据科学专业知识。
* **视觉处理：** Claude 3.5 Sonnet 擅长解释图表、图形和图像，准确转录文本以获取超出文本本身的洞察。
* **写作：** Claude 3.5 Sonnet 在理解细微差别和幽默方面有显著改进，产生高质量、真实且相关的内容。

如果您对我们的 Claude 模型系列进行基准比较感兴趣，请阅读我们的 [Claude 家族模型卡](https://www-cdn.anthropic.com/f2986af8d052f26236f6251da62d16172cfabd6e/claude-3-model-card.pdf) 获取更多信息。

### 能力展示

用一个演示来展示每个模型的各种能力是很困难的，但下面的函数尝试这样做。
我们让三个模型各自解决以下数学问题：

```
What is the geometric monthly fecal coliform mean of a distribution system with the following FC
 counts: 24, 15, 7, 16, 31 and 23? The result will be inputted into a NPDES DMR, therefore, round
 to the nearest whole number
```

**注意：正确答案是 18**

我们让每个模型解决这个数学问题 7 次并记录每次的答案：

In [19]:
def compare_model_capabilities():
    models = ["claude-3-5-sonnet-20240620", "claude-3-opus-20240229", "claude-3-sonnet-20240229", "claude-3-haiku-20240307"]
    task = """
    What is the geometric monthly fecal coliform mean of a distribution system with the following FC
 counts: 24, 15, 7, 16, 31 and 23? The result will be inputted into a NPDES DMR, therefore, round
 to the nearest whole number.  Respond only with a number and nothing else.
    """

    for model in models:
        answers = []
        for attempt in range(7):
            response = client.messages.create(
                model=model,
                max_tokens=1000,
                messages=[{"role": "user", "content": task}]
            )
            answers.append(response.content[0].text)

        print(f"Model: {model}")
        print(f"Answers: ", answers)

In [20]:
compare_model_capabilities()

Model: claude-3-5-sonnet-20240620
Answers:  ['18', '18', '18', '18', '18', '18', '18']
Model: claude-3-opus-20240229
Answers:  ['18', '18', '18', '18', '18', '18', '18']
Model: claude-3-sonnet-20240229
Answers:  ['17', '16', '17', '19', '18', '18', '18']
Model: claude-3-haiku-20240307
Answers:  ['17', '17', '18', '17', '17', '17', '18']


每个模型的确切输出会有所不同，但以下是单次运行的结果总结：

* `claude-3-5-sonnet-20240620` - 正确答案 **7/7** 次
* `claude-3-opus-20240229` - 正确答案 **7/7** 次
* `claude-3-sonnet-20240229` - 正确答案 **3/7** 次
* `claude-3-haiku-20240307` - 正确答案 **2/7** 次

显然，Claude 3.5 Sonnet 和 Claude 3 Opus 在这个特定的数学问题上表现最好。

**注意：这是对模型能力的一个非常简单的演示。绝不是一个严格的比较，仅作为易于理解的教育演示。请参阅 Claude 3 模型卡以获取更严格的、行业标准的基准比较。**

## 选择模型

下一个逻辑问题是：您应该使用哪个模型？在不了解给定应用程序的特定任务和需求的情况下，这是一个难以回答的问题。模型的选择会显著影响应用程序的性能、用户体验和成本效益：

* **能力**
  * 首先要考虑的是模型是否具备处理特定于您的应用程序的任务和用例的必要能力。不同的模型在不同领域有不同的性能水平，如通用语言理解、任务特定知识、推理能力和生成质量。必须将模型的优势与应用程序的需求相结合以确保最佳结果。
* **速度**
  * 模型处理和生成响应的速度是另一个关键因素，特别是对于需要实时或近实时交互的应用程序。更快的模型可以提供更具响应性和无缝的用户体验，减少延迟并提高整体可用性。然而，在速度和模型能力之间取得平衡很重要，因为最快的模型并不总是最适合您的特定需求。
* **成本**
  * 使用特定模型的相关成本是影响应用程序可行性和可扩展性的实际考虑因素。具有更高能力的模型通常伴随着更高的价格标签，包括 API 使用成本和所需的计算资源。评估不同模型的成本影响并确定仍能满足应用程序要求的最具成本效益的选项至关重要。

#### 一种方法：从 Haiku 开始

在实验时，我们经常建议从 Haiku 模型开始。Haiku 是一个轻量级且快速的模型，可以作为许多应用程序的绝佳起点。其速度和成本效益使其成为初步实验和原型制作的吸引人选项。在许多用例中，Haiku 被证明完全能够生成满足应用程序需求的高质量响应。通过从 Haiku 开始，您可以快速迭代您的应用程序、测试不同的提示词和配置，并在不产生重大成本或延迟的情况下评估模型的性能。如果对响应不满意，可以轻松"升级"到 Claude 3.5 Sonnet 等模型。


#### 评估和升级
随着您开发和优化应用程序，建立一套针对您的用例和提示词的全面评估至关重要。这些评估将作为衡量所选模型性能的基准，并帮助您做出关于潜在升级的明智决策。

如果您发现 Haiku 的响应不满足您的应用程序要求，或者您需要更高水平的复杂性和准确性，可以轻松过渡到 Sonnet 或 Opus 等更强大的模型。这些模型提供增强的能力，可以处理更复杂的任务和细微的语言理解。

通过建立严格的评估框架，您可以客观地比较不同模型在您的特定用例中的性能。这些经验证据将指导您的决策过程，确保您选择最符合您应用程序需求的模型。

***